
이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.




# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu130
CUDA version: 13.0
cuDNN version: 91200


In [2]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen3-VL-32B-Instruct"
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

# 학습데이터 200개만 추출
# train_df = train_df.sample(n=1000, random_state=SEED).reset_index(drop=True)

Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [ ]:
# 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    # 이미지 크기 변경 학습 속도 느려짐
    min_pixels=256*28*28,
    max_pixels=1536*28*28,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1058 [00:00<?, ?it/s]

trainable params: 536,870,912 || all params: 33,894,260,976 || trainable%: 1.5840


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
SYSTEM_INSTRUCT = (
    "You are an expert visual analysis assistant. "
    "Your task is to solve a multiple-choice question based on the provided image. "
    "Evaluate all choices equally and objectively. Do not favor any specific letter. " # 편향 방지 문구 추가
    "You must output ONLY a single lowercase letter representing the correct choice (a, b, c, or d). "
    "Do not include any explanations, punctuation, or additional text."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        "주어진 사진을 꼼꼼하게 관찰하고, 질문에 대한 가장 알맞은 정답을 고르세요.\n"
        "특히 사진 속 객체의 종류, 개수, 재질을 주의 깊게 분석해야 합니다.\n\n"
        f"질문: {question}\n\n"
        "선택지:\n"
        f"a. {a}\n"
        f"b. {b}\n"
        f"c. {c}\n"
        f"d. {d}\n\n"
        "위 선택지 중 가장 적절한 정답의 알파벳 소문자(a, b, c, d 중 하나)만 정확하게 출력하세요."
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

# 업 샘플링 희소 데이터 추가 학습

In [ ]:
import random
import pandas as pd
from collections import Counter
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from dataclasses import dataclass
from typing import Any
from sklearn.model_selection import train_test_split

# 1. [데이터 준비] 희소 정답 업샘플링 (이름표 포함)
def upsample_rare_answers_full(df, threshold=5, factor=2, seed=42):
    df = df.copy()
    df['is_copy'] = False

    # 실제 정답 텍스트를 기준으로 카운트
    answer_texts = [str(row[row["answer"]]).strip() for _, row in df.iterrows()]
    counter = Counter(answer_texts)

    # 희귀 정답 데이터 추출 및 복제 표시
    rare_mask = df.apply(lambda row: counter[str(row[row["answer"]]).strip()] <= threshold, axis=1)
    rare_df = df[rare_mask].copy()

    if len(rare_df) == 0: return df

    rare_df['is_copy'] = True

    # 전체 데이터 + 복제본 합치기 및 셔플
    upsampled = pd.concat([df, rare_df]).sample(frac=1, random_state=seed)
    return upsampled.reset_index(drop=True)

# --- [수정] 모든 에폭을 커버하는 통합 Dataset 클래스 ---
class VQAMCTargetedDataset(Dataset):
    def __init__(self, df, processor, transform=None, train=True, force_augment=False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.transform = transform
        self.train = train
        self.force_augment = force_augment

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        # [증강 로직] 2에폭은 무조건, 1에폭은 복제본만
        if self.train and self.transform:
            if self.force_augment or row.get('is_copy', False):
                img = self.transform(img)

        q = str(row["question"])
        choices = {'a': str(row["a"]), 'b': str(row["b"]), 'c': str(row["c"]), 'd': str(row["d"])}

        # 선지 실시간 셔플링
        orig_ans_key = str(row["answer"]).strip().lower()
        correct_text = choices[orig_ans_key]
        choice_values = list(choices.values())
        random.shuffle(choice_values)

        new_choices = {'a': choice_values[0], 'b': choice_values[1], 'c': choice_values[2], 'd': choice_values[3]}
        gold = [k for k, v in new_choices.items() if v == correct_text][0]
        user_text = build_mc_prompt(q, new_choices['a'], new_choices['b'], new_choices['c'], new_choices['d'])

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": user_text}]},
            {"role": "assistant", "content": [{"type": "text", "text": gold}]}
        ]
        return {"messages": messages, "image": img}


# --- [STEP 3] DataCollator (마스킹 유지) ---
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts = [self.processor.apply_chat_template(s["messages"], tokenize=False, add_generation_prompt=not self.train) for s in batch]
        images = [s["image"] for s in batch]
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")

        if self.train:
            labels = enc["input_ids"].clone()
            labels[enc["attention_mask"] == 0] = -100 # 패딩 마스킹
            image_pad_token_id = self.processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
            if image_pad_token_id is not None:
                labels[labels == image_pad_token_id] = -100 # 이미지 토큰 마스킹
            enc["labels"] = labels
        return enc

# --- [STEP 4] 실행 (Data Split -> Upsample -> Loaders) ---
train_full_upsampled = upsample_rare_answers_full(train_df, threshold=5, factor=2)

# 2. 증강 세팅
rare_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2)
])

full_transform_ep2 = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
])

# 3. 로더 생성
train_loader_ep1 = DataLoader(
    VQAMCTargetedDataset(train_full_upsampled, processor, transform=rare_transform, force_augment=False),
    batch_size=4, shuffle=True, collate_fn=DataCollator(processor)
)

train_loader_ep2 = DataLoader(
    VQAMCTargetedDataset(train_full_upsampled, processor, transform=full_transform_ep2, force_augment=True),
    batch_size=4, shuffle=True, collate_fn=DataCollator(processor)
)

print(f"🔥 학습 시작 준비 완료! 전체 데이터 샘플 수: {len(train_full_upsampled)}")

🔥 학습 시작 준비 완료! 전체 데이터 샘플 수: 6504


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
print(f"원본 데이터 개수: {len(train_df)}")
print(f"업샘플링 후 데이터 개수: {len(train_full_upsampled)}")
print(f"로더에 들어간 배치 개수(Step): {len(train_loader_ep1)}")

원본 데이터 개수: 5073
업샘플링 후 데이터 개수: 6504


NameError: name 'train_loader_ep1' is not defined

In [ ]:
import math
import bitsandbytes as bnb
from tqdm.auto import tqdm

GRAD_ACCUM = 4

# 💡 VRAM 다이어트 핵심: 8비트 AdamW 사용!
optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=2e-4, weight_decay=0.008)

# 총 2에폭 스텝 수 계산
num_training_steps = math.ceil(len(train_loader_ep1) / GRAD_ACCUM) * 2

# 코사인 스케줄러 적용 (후반부 미세 조정을 위해 코사인 추천)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.05),
    num_training_steps=num_training_steps
)

# 학습 루프 시작!
global_step = 0
for epoch in range(2): # 👈 2 에폭
    running = 0.0

    # 💡 [핵심] 에폭에 따라 데이터로더 교체! (1에폭: 원본, 2에폭: 100% 반전+컬러)
    current_loader = train_loader_ep1 if epoch == 0 else train_loader_ep2

    progress_bar = tqdm(current_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    model.train()
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
            running = 0.0

    if len(current_loader) % GRAD_ACCUM != 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

    # --- 검증 루프 (필요시 활성화) ---
    # model.eval()
    # val_loss = 0.0
    # val_steps = 0
    # with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
    #     for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
    #         vb = {k: v.to(device) for k, v in vb.items()}
    #         val_loss += model(**vb).loss.item()
    #         val_steps += 1
    # print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")

# 모델 저장 (폴더명 변경 확인)
SAVE_DIR = "/content/qwen3_ver_r_64_upsampling_3"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)

Epoch 1 [train]:   0%|          | 0/1626 [00:00<?, ?batch/s]

/tmp/ipykernel_76398/147872525.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch 2 [train]:   0%|          | 0/1626 [00:00<?, ?batch/s]

Saved: /content/qwen3_ver_r_64_upsampling


In [ ]:
import shutil
import os

# 1. 저장된 모델 폴더 경로 (수정된 경로)
SAVE_DIR = "/content/qwen3_ver_r_64_upsampling_3"

# 2. 압축 파일 이름 설정 (구분을 위해 r_64 명시)
backup_name = "/content/qwen3_r64_upsampling_backup_3"

# 3. 모델 폴더를 하나의 zip 파일로 압축하기
# (결과물: /content/qwen3_r64_upsampling_backup.zip 생성)
print(f"📦 '{SAVE_DIR}' 폴더 압축을 시작합니다... (용량에 따라 시간이 걸릴 수 있습니다)")
shutil.make_archive(backup_name, 'zip', SAVE_DIR)

# 4. 압축된 zip 파일을 구글 드라이브로 안전하게 복사
# (드라이브 내 저장 파일명: qwen3_r64_upsampling_backup.zip)
drive_path = "/content/drive/MyDrive/qwen3_r64_upsampling_backup.zip"

print(f"🚀 구글 드라이브로 복사 중: {drive_path}")
shutil.copy(f"{backup_name}.zip", drive_path)

print("✅ 모델 압축 및 구글 드라이브 백업이 완벽하게 완료되었습니다!")

📦 '/content/qwen3_ver_r_64_upsampling_3' 폴더 압축을 시작합니다... (용량에 따라 시간이 걸릴 수 있습니다)
🚀 구글 드라이브로 복사 중: /content/drive/MyDrive/qwen3_r64_upsampling_backup.zip
✅ 모델 압축 및 구글 드라이브 백업이 완벽하게 완료되었습니다!


# 전체 학습

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
!unzip "/content/drive/MyDrive/qwen3_r64_upsampling_backup.zip" -d "/content/qwen3_r64"

Archive:  /content/drive/MyDrive/qwen3_r64_upsampling_backup.zip
  inflating: /content/qwen3_r64/adapter_config.json  
  inflating: /content/qwen3_r64/README.md  
  inflating: /content/qwen3_r64/adapter_model.safetensors  
  inflating: /content/qwen3_r64/tokenizer.json  
  inflating: /content/qwen3_r64/chat_template.jinja  
  inflating: /content/qwen3_r64/tokenizer_config.json  
  inflating: /content/qwen3_r64/processor_config.json  


In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 기존에 쓰시던 Qwen3 클래스 임포트 (기존 코드에 있던 것 그대로 쓰시면 됩니다)
# from XXX import Qwen3VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen3-VL-32B-Instruct" # 예: "Qwen/Qwen..."
ADAPTER_PATH = "/content/qwen3_r64"

print("1️⃣ 양자화 설정 및 프로세서 로드...")
# 학습 때와 똑같이 4-bit 양자화 적용 (필수!)
print("1️⃣ 양자화 설정 및 프로세서 로드...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256*28*28,
    max_pixels=1536*28*28,
    trust_remote_code=True,
)

print("2️⃣ 베이스 모델 로드 (양자화 적용)...")
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

print("3️⃣ 93점 LoRA 가중치 결합 (이어서 학습하기 위한 설정)...")
# 💡 [필수 1] 양자화된 베이스 모델을 학습 가능한 상태로 준비시킵니다.
base_model = prepare_model_for_kbit_training(base_model)

# 💡 [필수 2] 기존 어댑터 가중치를 불러오되, 업데이트가 가능하도록 is_trainable=True 설정!
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, is_trainable=True)

# 💡 [필수 3] 추론 모드가 아닌 '학습 모드'로 설정!
model.train()
print("✅ 모델 로드 완료! 이제 벼락치기 추가 학습을 돌릴 수 있습니다.")

In [ ]:
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

In [ ]:
import random
import pandas as pd
from collections import Counter
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from dataclasses import dataclass
from typing import Any
from sklearn.model_selection import train_test_split

# 1. [데이터 준비] 희소 정답 업샘플링 (이름표 포함)
def upsample_rare_answers_full(df, threshold=5, factor=2, seed=42):
    df = df.copy()
    df['is_copy'] = False

    # 실제 정답 텍스트를 기준으로 카운트
    answer_texts = [str(row[row["answer"]]).strip() for _, row in df.iterrows()]
    counter = Counter(answer_texts)

    # 희귀 정답 데이터 추출 및 복제 표시
    rare_mask = df.apply(lambda row: counter[str(row[row["answer"]]).strip()] <= threshold, axis=1)
    rare_df = df[rare_mask].copy()

    if len(rare_df) == 0: return df

    rare_df['is_copy'] = True

    # 전체 데이터 + 복제본 합치기 및 셔플
    upsampled = pd.concat([df, rare_df]).sample(frac=1, random_state=seed)
    return upsampled.reset_index(drop=True)

# --- [STEP 2] 모든 데이터에 증강을 적용할 수 있는 통합 Dataset 클래스 ---
class VQAMCTargetedDataset(Dataset):
    def __init__(self, df, processor, transform=None, train=True, force_augment=False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.transform = transform
        self.train = train
        self.force_augment = force_augment

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        # 💡 [핵심 수정] force_augment가 True면 복제본 여부 상관없이 무조건 transform(랜덤) 적용
        if self.train and self.transform:
            if self.force_augment or row.get('is_copy', False):
                img = self.transform(img)

        q = str(row["question"])
        choices = {'a': str(row["a"]), 'b': str(row["b"]), 'c': str(row["c"]), 'd': str(row["d"])}

        # 선지 실시간 셔플링
        orig_ans_key = str(row["answer"]).strip().lower()
        correct_text = choices[orig_ans_key]
        choice_values = list(choices.values())
        random.shuffle(choice_values)

        new_choices = {'a': choice_values[0], 'b': choice_values[1], 'c': choice_values[2], 'd': choice_values[3]}
        gold = [k for k, v in new_choices.items() if v == correct_text][0]
        user_text = build_mc_prompt(q, new_choices['a'], new_choices['b'], new_choices['c'], new_choices['d'])

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": user_text}]},
            {"role": "assistant", "content": [{"type": "text", "text": gold}]}
        ]
        return {"messages": messages, "image": img}

# --- [STEP 3] DataCollator (마스킹 유지) ---
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts = [self.processor.apply_chat_template(s["messages"], tokenize=False, add_generation_prompt=not self.train) for s in batch]
        images = [s["image"] for s in batch]
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")

        if self.train:
            labels = enc["input_ids"].clone()
            labels[enc["attention_mask"] == 0] = -100 # 패딩 마스킹
            image_pad_token_id = self.processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
            if image_pad_token_id is not None:
                labels[labels == image_pad_token_id] = -100 # 이미지 토큰 마스킹
            enc["labels"] = labels
        return enc

# --- [STEP 4] 실행 (Data Split -> Upsample -> Loaders) ---
train_full_upsampled = upsample_rare_answers_full(train_df, threshold=5, factor=2)

# 💡 [단일 에폭 증강 세팅]
# p=0.5를 주어 매번 50% 확률로만 뒤집히게 만들어서, 원본과 변형을 골고루 보게 함
single_epoch_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2)
])

# 💡 [로더 생성] force_augment=True 로 설정하여 모든 데이터가 transform을 거치도록 강제함
train_loader = DataLoader(
    VQAMCTargetedDataset(train_full_upsampled, processor, transform=single_epoch_transform, force_augment=True),
    batch_size=4, shuffle=True, collate_fn=DataCollator(processor)
)

print(f"🔥 단일 에폭 벼락치기 학습 준비 완료! 전체 데이터 샘플 수: {len(train_full_upsampled)}")

In [ ]:
import math
import bitsandbytes as bnb
from tqdm.auto import tqdm
from transformers import get_cosine_schedule_with_warmup
import torch # 💡 필수!

# 💡 [추가 완료] device 설정!
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GRAD_ACCUM = 4

# VRAM 다이어트 핵심: 8비트 AdamW 사용!
optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=2e-4, weight_decay=0.008)

# 총 1에폭 스텝 수 계산
num_training_steps = math.ceil(len(train_loader) / GRAD_ACCUM)

# 코사인 스케줄러 적용
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.05),
    num_training_steps=num_training_steps
)

# 학습 루프 시작!
global_step = 0
for epoch in range(1): # 1 에폭 벼락치기
    running = 0.0

    current_loader = train_loader

    progress_bar = tqdm(current_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    model.train()
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()} # 👈 여기서 device를 사용합니다.

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
            running = 0.0

    # 남은 잔여 그래디언트 처리
    if len(current_loader) % GRAD_ACCUM != 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

# 모델 저장
SAVE_DIR = "/content/qwen3_ver_r_64_upsampling_3"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f"✅ 모델 저장 완료: {SAVE_DIR}")

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

# 아래는 좌우반전해서 이미지 보고 두번 추론한 결과 엮기

In [ ]:
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

# 💡 목표 토큰 ID 추출 ('a', 'b', 'c', 'd')
target_choices = ['a', 'b', 'c', 'd']
target_ids = [processor.tokenizer.encode(c, add_special_tokens=False)[0] for c in target_choices]

# 추론을 위해 평가 모드 전환
model.eval()
preds = []

# 추론 루프 시작 (TTA 적용)
for i in tqdm(range(len(test_df)), desc="Inference with TTA", unit="sample"):
    row = test_df.iloc[i]
    img_orig = Image.open(row["path"]).convert("RGB")
    img_flip = img_orig.transpose(Image.FLIP_LEFT_RIGHT)
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    # --- [원본 이미지 추론] ---
    text_orig = processor.apply_chat_template([
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [{"type": "image", "image": img_orig}, {"type": "text", "text": user_text}]}
    ], tokenize=False, add_generation_prompt=True)

    inputs_orig = processor(text=[text_orig], images=[img_orig], return_tensors="pt").to(device)

    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        outputs_orig = model(**inputs_orig)
        # 마지막 토큰의 로짓만 추출 후 즉시 입력 텐서 삭제 (VRAM 확보)
        choice_logits_orig = outputs_orig.logits[0, -1, target_ids].detach().cpu()

    # 💡 메모리 확보 핵심: 사용한 텐서 즉시 삭제
    del inputs_orig, outputs_orig
    torch.cuda.empty_cache()

    # --- [반전 이미지 추론] ---
    text_flip = processor.apply_chat_template([
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [{"type": "image", "image": img_flip}, {"type": "text", "text": user_text}]}
    ], tokenize=False, add_generation_prompt=True)

    inputs_flip = processor(text=[text_flip], images=[img_flip], return_tensors="pt").to(device)

    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        outputs_flip = model(**inputs_flip)
        choice_logits_flip = outputs_flip.logits[0, -1, target_ids].detach().cpu()

    # 💡 다시 메모리 확보
    del inputs_flip, outputs_flip
    # torch.cuda.empty_cache()

    # --- [앙상블 및 정답 결정] ---
    combined_logits = (choice_logits_orig + choice_logits_flip) / 2.0
    best_idx = combined_logits.argmax().item()
    preds.append(target_choices[best_idx])

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")

Inference with TTA:   0%|          | 0/5074 [00:00<?, ?sample/s]

/tmp/ipykernel_7084/2110056645.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipykernel_7084/2110056645.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):


In [ ]:
import shutil

# 이제 아래와 같은 함수들을 사용할 수 있습니다.
# shutil.copy('source.txt', 'destination.txt')
# shutil.rmtree('some_directory')
shutil.copy("/content/submission.csv", "/content/drive/MyDrive/submission_backup.csv")

print("구글 드라이브로 백업 완료! 이제 컴퓨터를 끄셔도 됩니다.")

구글 드라이브로 백업 완료! 이제 컴퓨터를 끄셔도 됩니다.
